# 04 - Model Experiments & Deep Learning Architecture

**Track 4 Lead**: Advik Goel (`advik_goel@berkeley.edu`)
**Course**: UC Berkeley DATASCI 207 (Machine Learning at Scale)
**Input**: `data/processed/processed_features.parquet` from Task 2 (307,511 rows x 153 columns)

Trains two TensorFlow models on the feature cohorts from notebook 03: a logistic regression (baseline) and a neural network. Then checks how they do for applicants with and without a credit bureau record, and for men vs. women, and tests Hypothesis 2: does adding alternative data help thin-file applicants more?

Use Restart & Run All so cell numbers start at 1. The next cell only does something in Colab.

In [ ]:
# NOTE FOR TEAMMATES:
# If you get 'Directory not found', add the shared folder to your My Drive:
# 1. Go to Google Drive (https://drive.google.com).
# 2. Click 'Shared with me' on the left menu.
# 3. Right-click the 'DATASCI207_finalproject' folder.
# 4. Select Organize -> Add shortcut (or press Shift + Z).
# 5. Choose 'My Drive' and click Add.

try:
    from google.colab import drive
    import shutil, os

    # Mount Google Drive
    drive.mount('/content/drive')

    # Copy shared project folder to local Colab storage
    src = "/content/drive/MyDrive/DATASCI207_finalproject"
    dst = "/content/DATASCI207_finalproject"

    shutil.copytree(src, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns('*.gdoc', '*.gsheet', '*.gslides'))
    print("Downloaded files:", os.listdir(dst))
except ImportError:
    print("Running in local environment; skipping Google Drive mount.")

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             f1_score, precision_recall_curve, roc_curve)

SEED = 1234
keras.utils.set_random_seed(SEED)   # seeds python, numpy and tensorflow

# Repo root (Jupyter may start in notebooks/), or the Drive copy in Colab
PROJECT_ROOT = Path().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if Path('/content/DATASCI207_finalproject').exists():
    PROJECT_ROOT = Path('/content/DATASCI207_finalproject')

OUT_DIR = PROJECT_ROOT / 'reports' / 'final_report'
FIG_DIR = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
df = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'processed_features.parquet')
df = df.replace([np.inf, -np.inf], np.nan)

print('Rows, columns:', df.shape)                    # should be (307511, 153)
print('Default rate:', round(df['TARGET'].mean(), 4))
print('Share with no bureau record:', round((df['HAS_BUREAU_HISTORY'] == 0).mean(), 4))

## 2. Cohorts and splits

Same cohorts and same 80/20 split as notebook 03, so the numbers can be compared directly.

| Cohort | Features |
|---|---|
| Cohort 1 - Traditional | Application + bureau |
| Cohort 2 - Alternative | Behavioral history + `EXT_SOURCE` scores |
| Cohort 3 - Hybrid | Everything + ratios |
| Cohort 4 - Thin-file style | Application + behavioral + ratios |

The neural net needs data to decide when to stop training, so 10% of the training rows are set aside for that ("early-stop set"). The validation set is only used for the final scores.

In [ ]:
NOT_F = ['SK_ID_CURR', 'TARGET']

CB_F = ['BUREAU_ACTIVE_LINES', 'BUREAU_TOTAL_LINES', 'BUREAU_TOTAL_OVERDUE_DEBT',
        'BUREAU_MAX_DAYS_OVERDUE', 'BUREAU_TOTAL_DEBT', 'BUREAU_OLDEST_CREDIT_DAYS',
        'HAS_BUREAU_HISTORY']

BEH_F = ['PREV_REFUSAL_COUNT', 'PREV_APPLICATION_COUNT', 'PREV_APPROVAL_RATE', 'PREV_MEAN_DOWN_PAYMENT_PCT',
         'POS_MONTH_COUNT', 'POS_MOST_RECENT_MONTH', 'POS_MEAN_DPD', 'POS_MAX_DPD', 'POS_MEAN_DPD_DEF',
         'POS_COMPLETED_COUNT', 'POS_ACTIVE_COUNT',
         'INST_DELAY_MEAN', 'INST_DELAY_MAX', 'INST_UNDERPAYMENT_FREQ', 'INST_TOTAL_PAYMENTS']

EXT_F = ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

RATIO_F = ['PAYMENT_RATE', 'INCOME_CREDIT_RATIO', 'ANNUITY_INCOME_RATIO']

STATIC_F = [c for c in df.columns if c not in NOT_F + CB_F + BEH_F + EXT_F + RATIO_F]

cohorts = {'Cohort 1': STATIC_F + CB_F,
           'Cohort 2': BEH_F + EXT_F,
           'Cohort 3': STATIC_F + CB_F + BEH_F + EXT_F + RATIO_F,
           'Cohort 4': STATIC_F + BEH_F + RATIO_F}

# same split as notebook 03
train_idx, val_idx = train_test_split(df.index, test_size=0.2,
                                      stratify=df['TARGET'], random_state=1234)

# 10% of train is used for early stopping and picking the threshold
fit_idx, es_idx = train_test_split(train_idx, test_size=0.1,
                                   stratify=df.loc[train_idx, 'TARGET'], random_state=SEED)

y_fit = df.loc[fit_idx, 'TARGET'].values
y_es  = df.loc[es_idx, 'TARGET'].values
y_val = df.loc[val_idx, 'TARGET'].values

print('Fit rows:       ', len(fit_idx))
print('Early-stop rows:', len(es_idx))
print('Validation rows:', len(val_idx))

## 3. Preprocessing

Similar to notebook 03: fill blanks with the training median, one-hot encode text columns, and scale everything with `StandardScaler`. Scaled values are clipped to [-5, 5] so a few extreme incomes don't throw off the neural net. Everything is learned from the training rows only.

In [ ]:
def prepare(features):
    # returns float32 arrays for the fit, early-stop and validation rows
    X_fit = df.loc[fit_idx, features].copy()
    X_es  = df.loc[es_idx, features].copy()
    X_val = df.loc[val_idx, features].copy()

    text_cols = [c for c in features if not pd.api.types.is_numeric_dtype(df[c])]
    num_cols  = [c for c in features if c not in text_cols]

    # numeric blanks -> training median
    medians = X_fit[num_cols].median()
    for X in (X_fit, X_es, X_val):
        X[num_cols] = X[num_cols].fillna(medians)

    # text blanks -> 'Missing', then one-hot encode
    for X in (X_fit, X_es, X_val):
        for c in text_cols:
            X[c] = X[c].astype('object').fillna('Missing')
    X_fit = pd.get_dummies(X_fit, columns=text_cols, dtype=float)
    X_es  = pd.get_dummies(X_es,  columns=text_cols, dtype=float).reindex(columns=X_fit.columns, fill_value=0.0)
    X_val = pd.get_dummies(X_val, columns=text_cols, dtype=float).reindex(columns=X_fit.columns, fill_value=0.0)

    # scale with the training mean / std, then clip outliers
    scaler = StandardScaler().fit(X_fit)
    out = [np.clip(scaler.transform(X), -5, 5).astype('float32') for X in (X_fit, X_es, X_val)]
    return out[0], out[1], out[2]


X_fit, X_es, X_val = prepare(cohorts['Cohort 3'])
print('Hybrid cohort after one-hot encoding:', X_fit.shape[1], 'columns')

## 4. Models

Logistic regression (baseline): a single `Dense(1, activation='sigmoid')` layer, which is exactly logistic regression written in TensorFlow. A small L2 penalty keeps the weights from blowing up.

Neural network: two hidden layers with ReLU, plus BatchNormalization and Dropout to reduce overfitting.

Both are trained with Adam and binary cross-entropy. Training stops when the early-stop AUC hasn't improved for 5 epochs, and the best epoch's weights are kept.

In [ ]:
def compile_model(model, lr=1e-3):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss='binary_crossentropy',
                  metrics=[keras.metrics.AUC(name='auc')])
    return model


def build_logreg(n_features):
    model = keras.Sequential([
        keras.Input(shape=(n_features,)),
        layers.Dense(1, activation='sigmoid', kernel_regularizer=keras.regularizers.l2(1e-4)),
    ])
    return compile_model(model)


def build_nn(n_features, units=128, dropout=0.3):
    model = keras.Sequential([
        keras.Input(shape=(n_features,)),
        layers.Dense(units, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(units // 2, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(1, activation='sigmoid'),
    ])
    return compile_model(model)


def train(model, X_fit, X_es, epochs=40):
    stop = keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=5,
                                         restore_best_weights=True)
    history = model.fit(X_fit, y_fit, validation_data=(X_es, y_es), epochs=epochs,
                        batch_size=1024, callbacks=[stop], verbose=0)
    return history.history


build_nn(X_fit.shape[1]).summary()

## 5. Tuning the neural network

Try a few sizes and dropout rates on the hybrid cohort and keep the one with the best early-stop AUC.

In [ ]:
configs = [dict(units=64,  dropout=0.2),
           dict(units=128, dropout=0.3),
           dict(units=256, dropout=0.3)]

tuning_rows, histories = [], []
for cfg in configs:
    keras.utils.set_random_seed(SEED)
    hist = train(build_nn(X_fit.shape[1], **cfg), X_fit, X_es)
    best = int(np.argmax(hist['val_auc']))
    histories.append(hist)
    tuning_rows.append({**cfg, 'best epoch': best + 1,
                        'train AUC': hist['auc'][best], 'early-stop AUC': hist['val_auc'][best]})

tuning = pd.DataFrame(tuning_rows)
display(tuning.round(4))

best_i = int(tuning['early-stop AUC'].idxmax())
BEST_NN = configs[best_i]
print('Best config:', BEST_NN)

In [ ]:
# learning curves for the best config: if the lines split apart, the model is overfitting
hist = histories[best_i]
epochs = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs, hist['loss'], label='train')
axes[0].plot(epochs, hist['val_loss'], label='early-stop set')
axes[0].set(title='Loss', xlabel='Epoch')
axes[1].plot(epochs, hist['auc'], label='train')
axes[1].plot(epochs, hist['val_auc'], label='early-stop set')
axes[1].set(title='ROC-AUC', xlabel='Epoch')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
fig.suptitle(f'Neural network learning curves (hybrid cohort, {BEST_NN})')
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_nn_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Train both models on every cohort

The decision threshold is the one that gives the best F1 on the early-stop set, the same rule as notebook 03.

Metrics on the validation set:
- ROC-AUC and PR-AUC: how well the model ranks defaulters above non-defaulters.
- Brier score: average squared error of the predicted probabilities (lower is better).
- F1, % flagged and % of defaults caught at the threshold.
- Predicted vs. actual loss: sum of (probability x loan amount) compared with the total amount of loans that actually defaulted. This assumes the whole loan is lost on default.

In [ ]:
def best_f1_threshold(y, p):
    precision, recall, thresholds = precision_recall_curve(y, p)
    f1 = 2 * precision * recall / (precision + recall + 1e-7)
    return thresholds[np.argmax(f1[:-1])]


def evaluate(y, p, threshold, credit):
    flagged = p >= threshold
    return {'ROC-AUC': roc_auc_score(y, p),
            'PR-AUC': average_precision_score(y, p),
            'Brier': brier_score_loss(y, p),
            'F1': f1_score(y, flagged),
            '% flagged': flagged.mean(),
            '% defaults caught': flagged[y == 1].mean(),
            'Predicted loss (M)': (p * credit).sum() / 1e6,
            'Actual loss (M)': (y * credit).sum() / 1e6}


credit_val = df.loc[val_idx, 'AMT_CREDIT'].values
preds, thresholds, results, models = {}, {}, [], {}

for cohort_name, features in cohorts.items():
    X_fit, X_es, X_val = prepare(features)
    for model_name in ['TF LogReg', 'TF Neural Net']:
        keras.utils.set_random_seed(SEED)
        if model_name == 'TF LogReg':
            model = build_logreg(X_fit.shape[1])
        else:
            model = build_nn(X_fit.shape[1], **BEST_NN)
        hist = train(model, X_fit, X_es)

        p_es  = model.predict(X_es,  batch_size=8192, verbose=0).ravel()
        p_val = model.predict(X_val, batch_size=8192, verbose=0).ravel()
        key = (model_name, cohort_name)
        preds[key], thresholds[key], models[key] = p_val, best_f1_threshold(y_es, p_es), model

        results.append({'Model': model_name, 'Cohort': cohort_name,
                        'Train AUC': max(hist['auc']),
                        **evaluate(y_val, p_val, thresholds[key], credit_val)})
        print(f"{model_name:<14} {cohort_name}: val ROC-AUC {results[-1]['ROC-AUC']:.4f}")

results = pd.DataFrame(results)
display(results.round(4))

In [ ]:
# validation ROC-AUC side by side, and how much the hybrid cohort adds over traditional
auc = results.pivot(index='Model', columns='Cohort', values='ROC-AUC')
auc['Hybrid - Traditional'] = auc['Cohort 3'] - auc['Cohort 1']
display(auc.round(4))

# for comparison, notebook 03's LightGBM on the hybrid cohort scored 0.7699

## 7. ROC and precision-recall curves (hybrid cohort)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for model_name in ['TF LogReg', 'TF Neural Net']:
    p = preds[(model_name, 'Cohort 3')]
    fpr, tpr, _ = roc_curve(y_val, p)
    precision, recall, _ = precision_recall_curve(y_val, p)
    axes[0].plot(fpr, tpr, label=f'{model_name} (AUC {roc_auc_score(y_val, p):.3f})')
    axes[1].plot(recall, precision, label=f'{model_name} (AP {average_precision_score(y_val, p):.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].axhline(y_val.mean(), color='k', ls='--', lw=1, label='default rate')
axes[0].set(title='ROC curve', xlabel='False positive rate', ylabel='True positive rate')
axes[1].set(title='Precision-recall curve', xlabel='Recall', ylabel='Precision')
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_roc_pr_hybrid.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Subgroups

Scores the validation set separately for:
- Group A: has a credit bureau record
- Group B: thin-file, no bureau record
- Group C: men vs. women

Same threshold for everyone. "% flagged" is the share of people the model would decline. If one group gets flagged much more often, or has fewer of its defaults caught, the model treats that group differently.

In [ ]:
val_df = df.loc[val_idx]
groups = {'A: bureau record': (val_df['HAS_BUREAU_HISTORY'] == 1).values,
          'B: thin-file':     (val_df['HAS_BUREAU_HISTORY'] == 0).values,
          'C: male':          (val_df['CODE_GENDER'] == 'M').values,
          'C: female':        (val_df['CODE_GENDER'] == 'F').values}

rows = []
for (model_name, cohort_name), p in preds.items():
    for group_name, mask in groups.items():
        rows.append({'Model': model_name, 'Cohort': cohort_name, 'Group': group_name,
                     'Rows': mask.sum(), 'Default rate': y_val[mask].mean(),
                     **evaluate(y_val[mask], p[mask], thresholds[(model_name, cohort_name)],
                                credit_val[mask])})

subgroups = pd.DataFrame(rows)
display(subgroups[subgroups['Cohort'] == 'Cohort 3'].drop(columns='Cohort').round(4))

In [ ]:
# ROC-AUC per group: traditional vs hybrid features, neural net
nn = subgroups[subgroups['Model'] == 'TF Neural Net'].pivot(index='Group', columns='Cohort', values='ROC-AUC')
nn = nn.loc[list(groups)]

x = np.arange(len(nn))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - 0.2, nn['Cohort 1'], 0.4, label='Traditional (Cohort 1)', color='#999999')
ax.bar(x + 0.2, nn['Cohort 3'], 0.4, label='Hybrid (Cohort 3)', color='#4C72B0')
ax.set_xticks(x, nn.index)
ax.set_ylim(nn.values.min() - 0.05, nn.values.max() + 0.03)
ax.set(ylabel='Validation ROC-AUC', title='Neural net ROC-AUC by group')
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_subgroup_auc.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Hypothesis 2: does alternative data help thin-file applicants more?

For each group, the "lift" is hybrid AUC minus traditional AUC. Hypothesis 2 says the lift is bigger for thin-file applicants (B) than for applicants with a bureau record (A).

To check that the difference isn't just luck, we bootstrap: resample the validation rows 1,000 times with replacement, recompute the difference each time, and look at the middle 95% of the results. If that whole range is above 0, the thin-file lift is reliably bigger.

In [ ]:
rows_a = np.where(groups['A: bureau record'])[0]
rows_b = np.where(groups['B: thin-file'])[0]
rng = np.random.default_rng(SEED)
N_BOOT = 1000

h2_rows, boot = [], {}
for model_name in ['TF LogReg', 'TF Neural Net']:
    p_trad = preds[(model_name, 'Cohort 1')]
    p_hyb  = preds[(model_name, 'Cohort 3')]

    def lift(rows):
        return roc_auc_score(y_val[rows], p_hyb[rows]) - roc_auc_score(y_val[rows], p_trad[rows])

    diffs = []
    for _ in range(N_BOOT):
        a = rng.choice(rows_a, size=len(rows_a), replace=True)
        b = rng.choice(rows_b, size=len(rows_b), replace=True)
        diffs.append(lift(b) - lift(a))
    diffs = np.array(diffs)
    boot[model_name] = diffs

    low, high = np.percentile(diffs, [2.5, 97.5])
    h2_rows.append({'Model': model_name, 'Lift A (bureau)': lift(rows_a), 'Lift B (thin-file)': lift(rows_b),
                    'B - A': lift(rows_b) - lift(rows_a), '95% range low': low, '95% range high': high,
                    'Share of resamples above 0': (diffs > 0).mean(),
                    'H2 supported?': 'yes' if low > 0 else 'no'})

h2 = pd.DataFrame(h2_rows)
display(h2.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for model_name, diffs in boot.items():
    ax.hist(diffs, bins=50, alpha=0.5, label=model_name)
ax.axvline(0, color='k', ls='--')
ax.set(title='Bootstrap: thin-file lift minus bureau lift',
       xlabel='Difference in ROC-AUC lift', ylabel='Count')
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'fig_h2_bootstrap.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save results

Tables go to `reports/final_report/`, figures to `reports/final_report/figures/`, and the hybrid neural net to `saved_models/`. In Colab, download them before the runtime ends.

In [ ]:
results.to_csv(OUT_DIR / 'nb04_results.csv', index=False)
subgroups.to_csv(OUT_DIR / 'nb04_subgroups.csv', index=False)
h2.to_csv(OUT_DIR / 'nb04_h2.csv', index=False)
tuning.to_csv(OUT_DIR / 'nb04_tuning.csv', index=False)

(PROJECT_ROOT / 'saved_models').mkdir(exist_ok=True)
models[('TF Neural Net', 'Cohort 3')].save(PROJECT_ROOT / 'saved_models' / 'tf_nn_hybrid.keras')
print('Saved to', OUT_DIR)